# DOE integrated full pipeline notebook

This notebook keeps the DOE three-workbook ML workflow in one Jupyter file for Anaconda desktops that do not have Git. It uses only standard Anaconda-style packages: pandas, numpy, scikit-learn, openpyxl, and joblib.

It writes outputs locally under `outputs_runtime/` and models under `models_runtime/`. Do not upload workbook rows, prediction rows, or fitted models to GitHub.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, re
import numpy as np
import pandas as pd

DATA_DIR = Path.home() / 'Downloads' / 'Northslopedatasets06052026'
WORKBOOKS = ('curated_dataset1.xlsx','curated_dataset2.xlsx','curated_dataset3.xlsx')
OUT = Path.cwd() / 'outputs_runtime'
MOD = Path.cwd() / 'models_runtime'
OUT.mkdir(exist_ok=True); MOD.mkdir(exist_ok=True)
print('Notebook folder:', Path.cwd())
print('Data folder:', DATA_DIR)
for f in WORKBOOKS:
    print(f, 'FOUND' if (DATA_DIR/f).exists() else 'MISSING')

REQ = ('well_alias','depth_m','gr_api','rt_ohm_m','rhob_g_cc')
CHONG = ('rhob_g_cc','density_porosity_vv','rt_ohm_m','gr_api','vp_km_s','vs_km_s')
ALIASES = {
 'well_alias':('well_alias','WELL','WELL_NAME','Well Name','UWI','API'),
 'depth_m':('depth_m','DEPTH_M','DEPTH','MD','MD_M','TVD','TVD_M','DEPT'),
 'gr_api':('gr_api','GR','GAMMA','GAMMA_RAY','Gamma Ray'),
 'rt_ohm_m':('rt_ohm_m','RT','ILD','RDEP','RES','RES_DEEP','Deep formation resistivity'),
 'rhob_g_cc':('rhob_g_cc','RHOB','DEN','DENSITY','Rho_b','Density_gcpcc','Density_gpcc'),
 'density_porosity_vv':('density_porosity_vv','DPHI','PHID','DEN_POR','Phi_porosity','phi_den'),
 'neutron_porosity_vv':('neutron_porosity_vv','NPHI','TNPH','NEUTRON_POR','phi_neut'),
 'dt_us_ft':('dt_us_ft','DT','DTC','AC'), 'dts_us_ft':('dts_us_ft','DTS','DTSM'),
 'vp_km_s':('vp_km_s','VP_KM_S'), 'vs_km_s':('vs_km_s','VS_KM_S'),
 'vp_m_s':('vp_m_s','Vp','VP','VELP','VP_M_S'), 'vs_m_s':('vs_m_s','Vs','VS','VS1','VELS','VS_M_S'),
 'nmr_porosity_vv':('nmr_porosity_vv','NMRPHI','TCMR','CMRP','phi_nmr'),
 'caliper_in':('caliper_in','CALI','CALIPER','caliper','CAL1'),
 'temperature_c':('temperature_c','TEMP','TEMPERATURE_C'), 'pressure_mpa':('pressure_mpa','PRESSURE','PRESSURE_MPA','PP_MPA')}
TARGET_ALIASES = {
 'hydrate_saturation':('Sgh','S_h','Sh','NMR_SAT','Hydrate Saturation','hydrate_saturation_vv'),
 'irreducible_or_residual_water_saturation':('Swr','S_wr','irreducible_water_saturation_vv'),
 'phase_or_occurrence_label':('interpreted phase label','phase_label','hydrate_phase','hydrate_occurrence_label','runtime_phase_label')}
CLASS_HINTS=('occurrence','class','label','phase','hydratepresent'); REG_HINTS=('saturation','sat','sgh','shyd','nmr_sat')
ID_CONTEXT={'well_alias','depth_m','source_dataset','dataset_file','source_sheet','split','row_index'}

def norm(x): return ''.join(c for c in str(x).lower() if c.isalnum())
def clean_label(x): return re.sub(r'[^A-Za-z0-9_.-]+','_',str(x).strip()).strip('._-')[:90] or 'run'
def clean_num(s):
    v=pd.to_numeric(s, errors='coerce').replace([np.inf,-np.inf], np.nan)
    return v.mask(v.abs()>1e12)
def target_lookup():
    d={}
    for fam, als in TARGET_ALIASES.items():
        for a in als: d[norm(a)]=(fam,a)
    return d
def is_target_col(c):
    n=norm(c)
    if n in target_lookup() or n in {'class','label','target','y','occurrence','phase','hydratepresent'}: return True
    return (('hydrate' in n or 'sgh' in n or n.startswith('sh')) and any(w in n for w in ('sat','saturation','occur','class','label','phase')))
def is_sat_col(c):
    n=norm(c)
    if any(w in n for w in ('porosity','density','sample','station','status')): return False
    return n in {'sgh','sh','shyd','hydratesat','hydratesaturation','nmrsat','swr','sw','swi','swirr'} or 'sat' in n or 'saturation' in n
def canonical(c):
    n=norm(c)
    for can, als in ALIASES.items():
        if n==norm(can) or any(n==norm(a) for a in als): return can
    return None
def standardize(df):
    look={norm(c):c for c in df.columns}; ren={}
    for can, als in ALIASES.items():
        for a in als:
            if norm(a) in look: ren[look[norm(a)]]=can; break
    out=df.rename(columns=ren).copy()
    for c in out.columns:
        if c!='well_alias':
            v=clean_num(out[c])
            if v.notna().any(): out[c]=v
    if 'well_alias' in out: out['well_alias']=out['well_alias'].astype(str)
    return out
def add_features(df):
    x=df.copy()
    if 'gr_api' in x: x['vshale']=((clean_num(x['gr_api'])-30)/(105-30)).clip(0,1)
    if 'rhob_g_cc' in x and 'density_porosity_vv' not in x: x['density_porosity_vv']=((2.65-clean_num(x['rhob_g_cc']))/(2.65-1.03)).clip(0,0.7)
    if 'dt_us_ft' in x and 'vp_km_s' not in x: x['vp_km_s']=304.8/clean_num(x['dt_us_ft'])
    if 'dts_us_ft' in x and 'vs_km_s' not in x: x['vs_km_s']=304.8/clean_num(x['dts_us_ft'])
    if 'vp_m_s' in x and 'vp_km_s' not in x: x['vp_km_s']=clean_num(x['vp_m_s'])/1000
    if 'vs_m_s' in x and 'vs_km_s' not in x: x['vs_km_s']=clean_num(x['vs_m_s'])/1000
    if {'vp_km_s','vs_km_s'} <= set(x.columns): x['vp_vs_ratio']=clean_num(x['vp_km_s'])/clean_num(x['vs_km_s'])
    if {'rhob_g_cc','vs_km_s'} <= set(x.columns): x['shear_modulus_gpa']=clean_num(x['rhob_g_cc'])*clean_num(x['vs_km_s'])**2
    if {'rhob_g_cc','vp_km_s','vs_km_s'} <= set(x.columns): x['bulk_modulus_gpa']=clean_num(x['rhob_g_cc'])*(clean_num(x['vp_km_s'])**2-(4/3)*clean_num(x['vs_km_s'])**2)
    if {'density_porosity_vv','nmr_porosity_vv'} <= set(x.columns): x['nmr_density_hydrate_proxy']=((clean_num(x['density_porosity_vv'])-clean_num(x['nmr_porosity_vv']))/clean_num(x['density_porosity_vv']).clip(.01)).clip(0,1)
    if {'density_porosity_vv','rt_ohm_m'} <= set(x.columns): x['archie_hydrate_proxy']=(1-((0.12/(clean_num(x['density_porosity_vv']).clip(.04)**2*clean_num(x['rt_ohm_m'])))**0.5)).clip(0,1)
    return x.replace([np.inf,-np.inf],np.nan)
def context_col(c):
    n=norm(c); can=canonical(c)
    return n.startswith('unnamed') or n in {'index','row','rowindex','depth','depthm','depthft','dept','md','tvd'} or 'unit' in n or 'depth' in n or can in {'well_alias','depth_m'}
def feature_matrix(df, target_cols):
    f=add_features(df); cols=[]; rows=[]
    for c in f.columns:
        name=str(c); can=canonical(name)
        if name in target_cols or is_target_col(name) or name in ID_CONTEXT or context_col(name): continue
        if can and can!=name and can in f.columns: continue
        v=clean_num(f[name]) if not pd.api.types.is_bool_dtype(f[name]) else f[name].astype(int)
        if v.notna().sum()==0: continue
        f[name]=v; cols.append(name); rows.append({'feature_column':name,'non_null_rows':int(v.notna().sum()),'coverage_fraction':round(float(v.notna().mean()),4),'minimum':float(v.min()),'maximum':float(v.max())})
    return f[cols].copy(), pd.DataFrame(rows)
def first_sheet(path, sheet=None):
    with pd.ExcelFile(path) as xl: names=list(xl.sheet_names)
    for s in ([sheet] if sheet else names):
        df=pd.read_excel(path, sheet_name=s)
        if not df.empty: return df, str(s)
    return pd.read_excel(path, sheet_name=names[0]), str(names[0])
def load_dataset(data_dir, file, split, sheet=None):
    raw,s=first_sheet(data_dir/file, sheet); df=standardize(raw)
    df['source_dataset']=Path(file).stem; df['dataset_file']=file; df['source_sheet']=s; df['split']=split; df['row_index']=np.arange(len(df))
    if 'well_alias' not in df: df['well_alias']=Path(file).stem
    return df

def scan_headers(data_dir=DATA_DIR, files=WORKBOOKS, out=OUT/'header_scan'):
    out.mkdir(parents=True, exist_ok=True); sheets=[]; cols=[]
    for f in files:
        p=data_dir/f
        if not p.exists(): continue
        with pd.ExcelFile(p) as xl: names=list(xl.sheet_names)
        for s in names:
            sample=pd.read_excel(p, sheet_name=s, nrows=25); head=pd.read_excel(p, sheet_name=s, nrows=0)
            sheets.append({'workbook':f,'sheet_name':s,'sampled_rows':len(sample),'column_count':len(head.columns),'has_target_like_header':any(is_target_col(c) for c in head.columns)})
            for i,c in enumerate(head.columns,1):
                vals=sample[c] if c in sample else pd.Series(dtype=object); nums=clean_num(vals); role='possible_target_review' if is_target_col(c) else ('identifier_or_depth_axis' if canonical(c) in {'well_alias','depth_m'} else ('candidate_feature_or_context' if canonical(c) else 'unmapped_review'))
                task='classification' if role=='possible_target_review' and any(h in norm(c) for h in CLASS_HINTS) else ('regression' if is_sat_col(c) else '')
                cols.append({'workbook':f,'sheet_name':s,'column_position':i,'original_header':str(c),'normalized_header':norm(c),'canonical_header':canonical(c) or '', 'role_hint':role,'non_null_sample_rows':int(vals.notna().sum()),'numeric_sample_rows':int(nums.notna().sum()),'unique_sample_values':int(vals.dropna().nunique()),'suggested_target_task':task})
    sf=pd.DataFrame(sheets); cf=pd.DataFrame(cols); tf=cf[cf.role_hint.str.contains('target',case=False,na=False)].copy() if not cf.empty else pd.DataFrame()
    sf.to_csv(out/'workbook_sheet_inventory.csv',index=False); cf.to_csv(out/'workbook_column_inventory.csv',index=False); tf.to_csv(out/'target_header_hints.csv',index=False)
    return {'run_dir':str(out),'target_hint_count':len(tf),'target_header_hints':str(out/'target_header_hints.csv')}

def choose_target(df, requested='auto', task='auto'):
    cands=[c for c in df.columns if is_target_col(c)]
    if requested!='auto': col=next((c for c in df.columns if norm(c)==norm(requested)), None)
    else: col=cands[0] if cands else None
    if not col: return None,'readiness_only','no target-like column detected'
    if task=='auto': task='classification' if any(h in norm(col) for h in CLASS_HINTS) else 'regression'
    return col,task,'selected'

def run_pipeline(data_dir=DATA_DIR, train_file='curated_dataset1.xlsx', test_files=('curated_dataset2.xlsx','curated_dataset3.xlsx'), target='auto', task='auto', run_label=None):
    from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import Pipeline
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, balanced_accuracy_score, f1_score
    from joblib import dump
    label=clean_label(run_label or 'ml_run_'+datetime.now().strftime('%Y%m%d_%H%M%S')); out=OUT/label; mod=MOD/label; out.mkdir(parents=True,exist_ok=True); mod.mkdir(parents=True,exist_ok=True)
    train=load_dataset(data_dir,train_file,'train'); tests=[load_dataset(data_dir,f,'test') for f in test_files if (data_dir/f).exists()]
    col,task,reason=choose_target(train,target,task); target_cols={str(c) for c in train.columns if is_target_col(c)}
    X,finv=feature_matrix(train,target_cols); finv.to_csv(out/'feature_columns.csv',index=False)
    pd.DataFrame([{'dataset':train_file,'split':'train','rows':len(train),'columns':len(train.columns)}]+[{'dataset':d.dataset_file.iloc[0],'split':'test','rows':len(d),'columns':len(d.columns)} for d in tests]).to_csv(out/'dataset_inventory.csv',index=False)
    if not col or X.empty:
        result={'status':'readiness_only','blocked_reason':reason if not col else 'no numeric feature columns','run_dir':str(out)}; (out/'run_manifest.json').write_text(json.dumps(result,indent=2)); return result
    y=clean_num(train[col]) if task=='regression' else train[col].astype('string'); mask=y.notna() & X.notna().any(axis=1); Xtr=X.loc[mask]; ytr=y.loc[mask]; rows=train.loc[mask].reset_index(drop=True)
    if len(Xtr)<3: result={'status':'readiness_only','blocked_reason':'fewer than 3 training rows','run_dir':str(out)}; (out/'run_manifest.json').write_text(json.dumps(result,indent=2)); return result
    model=Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',MinMaxScaler()),('model',RandomForestRegressor(n_estimators=250,min_samples_leaf=2,random_state=42) if task=='regression' else RandomForestClassifier(n_estimators=250,min_samples_leaf=2,class_weight='balanced',random_state=42))])
    model.fit(Xtr,ytr); pred=model.predict(Xtr)
    metrics=[]
    if task=='regression': metrics.append({'dataset':train_file,'split':'train','mae':float(mean_absolute_error(ytr,pred)),'rmse':float(np.sqrt(mean_squared_error(ytr,pred))),'r2':float(r2_score(ytr,pred)) if len(ytr)>1 else None})
    else: metrics.append({'dataset':train_file,'split':'train','accuracy':float(accuracy_score(ytr,pred)),'balanced_accuracy':float(balanced_accuracy_score(ytr,pred)),'f1_macro':float(f1_score(ytr,pred,average='macro',zero_division=0))})
    pd.DataFrame({'source_dataset':rows.source_dataset,'dataset_file':rows.dataset_file,'well_alias':rows.well_alias,'row_index':rows.row_index,'target_column':col,'y_true':ytr.reset_index(drop=True),'y_pred':pred}).to_csv(out/f'predictions_{Path(train_file).stem}.csv',index=False)
    for d in tests:
        Xt,_=feature_matrix(d,target_cols);
        for f in X.columns:
            if f not in Xt: Xt[f]=np.nan
        Xt=Xt[X.columns]; m=Xt.notna().any(axis=1); Xt2=Xt.loc[m]; src=d.loc[m].reset_index(drop=True)
        if Xt2.empty: continue
        p=model.predict(Xt2); pf=pd.DataFrame({'source_dataset':src.source_dataset,'dataset_file':src.dataset_file,'well_alias':src.well_alias,'row_index':src.row_index,'target_column':col,'y_pred':p})
        if col in src.columns:
            yt=clean_num(src[col]) if task=='regression' else src[col].astype('string'); pf['y_true']=yt
        pf.to_csv(out/f'predictions_{Path(str(d.dataset_file.iloc[0])).stem}.csv',index=False)
    pd.DataFrame(metrics).to_csv(out/'train_metrics.csv',index=False); dump({'model':model,'features':list(X.columns),'target':col,'task':task}, mod/'model.joblib')
    result={'status':'trained','run_dir':str(out),'model_dir':str(mod),'target':col,'task':task,'feature_count':len(X.columns)}; (out/'run_manifest.json').write_text(json.dumps(result,indent=2)); return result

def run_all_saturation_targets(data_dir=DATA_DIR, out=OUT/'all_saturation_targets'):
    out.mkdir(parents=True,exist_ok=True); rows=[]
    for wb in WORKBOOKS:
        p=data_dir/wb
        if not p.exists(): continue
        with pd.ExcelFile(p) as xl: sheets=xl.sheet_names
        for sh in sheets:
            raw=pd.read_excel(p,sheet_name=sh); df=standardize(raw); sats=[c for c in df.columns if is_sat_col(c)]
            for t in sats:
                res=run_pipeline(data_dir, train_file=wb, test_files=tuple(f for f in WORKBOOKS if f!=wb), target=t, task='regression', run_label='sat_'+clean_label(wb+'_'+str(sh)+'_'+str(t)))
                rows.append({'workbook':wb,'sheet_name':sh,'target_column':t,**res})
    pd.DataFrame(rows).to_csv(out/'run_summary.csv',index=False); return {'status':'complete','output_dir':str(out),'target_runs':len(rows)}
print('Functions loaded.')

## 1. Header scan
Run this first. Then inspect the displayed target hints and decide which exact target header you want.

In [ ]:
header_result = scan_headers()
print(json.dumps(header_result, indent=2))
target_hints = pd.read_csv(header_result['target_header_hints']) if Path(header_result['target_header_hints']).exists() else pd.DataFrame()
display(target_hints.head(40))

## 2. Main pipeline
Change `TARGET` to the exact target column, or leave it as `auto`. Default training file is dataset 1; default prediction/test files are datasets 2 and 3.

In [ ]:
TARGET = 'auto'      # or exact header from target_hints
TASK = 'auto'         # auto, regression, or classification
main_result = run_pipeline(target=TARGET, task=TASK, run_label='notebook_main_pipeline')
print(json.dumps(main_result, indent=2, default=str))
print('Open output folder:', main_result.get('run_dir'))

## 3. Optional: dataset 3 as the labeled training file
Run this if the target labels are only in `curated_dataset3.xlsx`.

In [ ]:
TARGET_DATASET3 = 'auto'
dataset3_result = run_pipeline(train_file='curated_dataset3.xlsx', test_files=('curated_dataset1.xlsx','curated_dataset2.xlsx'), target=TARGET_DATASET3, task='auto', run_label='notebook_dataset3_training')
print(json.dumps(dataset3_result, indent=2, default=str))
print('Open output folder:', dataset3_result.get('run_dir'))

## 4. Optional: run every saturation-like target
This treats all saturation-like columns as target-only outputs and runs separate regressions.

In [ ]:
multi_result = run_all_saturation_targets()
print(json.dumps(multi_result, indent=2, default=str))
summary_path = Path(multi_result['output_dir']) / 'run_summary.csv'
if summary_path.exists(): display(pd.read_csv(summary_path).head(50))